# 02 - LLM + RAG test (Member C)

Tests the knowledge-enhanced LLM step (paper Section 3.4, Fig. 10) with a local model through Ollama's
OpenAI-compatible API.

1. Does switching off Qwen's reasoning ("thinking") work, and how much faster is it?
2. **(a)** The paper's two Fig. 10 cases, 3 runs each, with the paper's own context.
3. **(b)** Three made-up rules with our dataset's column names, through the full pipeline (our knowledge base + retrieval), with and without RAG.
4. **(c)** Time estimate for 300 rules x 3 runs x 2 settings.

Requirements: Ollama running with the model in `.env` (`LLM_MODEL`), `pip install -r requirements.txt`.

In [1]:
import sys, time
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
(ROOT / "results" / "tables").mkdir(parents=True, exist_ok=True)

from src import rag
from src import llm_diagnose as ld

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)
print({k: v for k, v in ld.SETTINGS.items() if k != "api_key"})


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\ASUS\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\ASUS\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\ASUS\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\ASUS\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\ASUS\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\ASUS\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



{'base_url': 'http://localhost:11434/v1', 'model': 'qwen3.5:4b', 'temperature': 0.7, 'top_p': 0.7, 'runs': 3, 'max_reason_words': 60}


## 1. Switching reasoning off

Same prompt (paper Case 2) sent with three settings. `reasoning chars` is the length of the separate
`reasoning` field Ollama returns; `<think> in content` checks whether thinking leaked into the answer text.

In [2]:
case2_prompt = ld.build_prompt(
    "Antecedent: Chilled water supply and return temperature difference <2 C; "
    "Consequent: Input power <100 kW; Support: 0.0491; Confidence: 0.5032",
    ["Rated chilled water supply temperature: 7 C", "Rated chilled water return temperature: 12 C",
     "The temperature difference between the refrigeration supply and return water is less than 2 C, "
     "and the heat exchange is insufficient"])

def raw_call(messages, **extra):
    t0 = time.perf_counter()
    r = ld._get_client().chat.completions.create(
        model=ld.SETTINGS["model"], messages=messages, temperature=0.7, top_p=0.7,
        response_format={"type": "json_object"}, **extra)
    msg = r.choices[0].message
    reasoning = getattr(msg, "reasoning", None) or (msg.model_extra or {}).get("reasoning") or ""
    return {"latency_s": round(time.perf_counter() - t0, 1), "completion_tokens": r.usage.completion_tokens,
            "reasoning chars": len(reasoning), "<think> in content": "<think>" in (msg.content or "")}

no_think_msgs = [case2_prompt[0], {"role": "user", "content": case2_prompt[1]["content"] + " /no_think"}]
think_test = pd.DataFrame([
    {"method": "default (thinking on)", **raw_call(case2_prompt)},
    {"method": 'extra_body={"think": False}', **raw_call(case2_prompt, extra_body={"think": False})},
    {"method": "/no_think in user message", **raw_call(no_think_msgs)},
    {"method": 'reasoning_effort="none" (used)', **raw_call(case2_prompt, **ld.NO_THINKING)},
])
think_test

,method,latency_s,completion_tokens,reasoning chars,<think> in content
0,default (thinking on),88.8,3887,16325,False
1,"extra_body={""think"": False}",76.0,3887,16453,False
2,/no_think in user message,75.0,3883,16368,False
3,"reasoning_effort=""none"" (used)",1.8,74,0,False


## (a) Paper Fig. 10 cases, 3 runs each

Rules and context text exactly as in the paper. The [User] prompt has one added sentence: "validity is true only if the rule indicates an anomaly (labels 1-3); for label 0 it is false." Expected: Case 1 validity false / label 0,
Case 2 validity true / label 2.

In [2]:
cases = [
    {"case": 1,
     "rule": {"antecedent": ("Cooling capacity=[1500,2300] kW",), "consequent": ("System COP=[4,6]",),
              "support": 0.2503, "confidence": 0.927, "lift": None},
     "context": ["Rated cooling capacity: 2800 kW", "Rated COP: 5.92",
                 "System COP is in the normal range of 4-6",
                 "Cooling capacity is 1500-2300 kW, normal load range"],
     "expected": (False, 0)},
    {"case": 2,
     "rule": {"antecedent": ("Chilled water supply and return temperature difference <2 C",),
              "consequent": ("Input power <100 kW",), "support": 0.0491, "confidence": 0.5032, "lift": None},
     "context": ["Rated chilled water supply temperature: 7 C", "Rated chilled water return temperature: 12 C",
                 "The temperature difference between the refrigeration supply and return water is less "
                 "than 2 C, and the heat exchange is insufficient"],
     "expected": (True, 2)},
]

rows = []
for c in cases:
    prompt = ld.build_prompt(ld.rule_to_text(c["rule"]), c["context"])
    for run in range(1, ld.SETTINGS["runs"] + 1):
        r = ld.classify_rule(prompt)
        rows.append({"case": c["case"], "run": run, "validity": r["validity"], "label": r["label"],
                     "reason": r["reason"] or r["error"],
                     "matches paper": (r["validity"], r["label"]) == c["expected"],
                     "latency_s": round(r["latency_s"], 2), "tokens": r["total_tokens"], "failed": r["failed"]})
paper_cases = pd.DataFrame(rows)
paper_cases.to_csv(ROOT / "results/tables/llm_paper_cases.csv", index=False)
print("Prompt sent for Case 2:\n")
print(ld.build_prompt(ld.rule_to_text(cases[1]["rule"]), cases[1]["context"])[1]["content"])
paper_cases

Prompt sent for Case 2:

[Rule]
Antecedent: Chilled water supply and return temperature difference <2 C; Consequent: Input power <100 kW; Support: 0.0491; Confidence: 0.5032

[Context]
- Rated chilled water supply temperature: 7 C
- Rated chilled water return temperature: 12 C
- The temperature difference between the refrigeration supply and return water is less than 2 C, and the heat exchange is insufficient

[User]
Please reply in JSON with three fields: 1. "validity": true or false; 2. "label": one of {"0 Invalid rules", "1 Design anomaly", "2 Operational anomaly", "3 Performance degradation"}; 3. "reason": concise (<=60 words) explanation. validity is true only if the rule indicates an anomaly (labels 1-3); for label 0 it is false.


,case,run,validity,label,reason,matches paper,latency_s,tokens,failed
0,1,1,False,0,"The rule describes normal operational parameters (cooling capacity 1500-2300 kW and COP 4-6) matching the system's rated specs. It does not indicate any anomaly or degradation, making it an invalid rule for this classification scheme.",True,14.23,327,False
1,1,2,False,0,The rule describes normal operational parameters where cooling capacity and COP fall within expected ranges. It does not indicate any anomaly or degradation.,True,1.11,298,False
2,1,3,False,0,The rule describes normal operational conditions (cooling capacity and COP within expected ranges) rather than an anomaly or degradation. It lacks a deviation from rated performance to qualify as a design or operational anomaly.,True,1.37,312,False
3,2,1,False,0,"The antecedent describes a normal or suboptimal condition (low delta T), but the consequent (low power) contradicts typical physics where insufficient heat exchange requires higher compressor work to maintain flow. Low power with low delta T likely indicates system under-sizing or idle operation, not a specific design/operational anomaly requiring intervention.",False,2.06,330,False
4,2,2,True,2,"A supply-return temperature difference under 2°C indicates insufficient heat exchange, suggesting a system fault rather than normal operation or design intent.",True,1.10,290,False
5,2,3,False,0,"The rule suggests low power consumption (<100 kW) as a consequence of insufficient temperature difference. However, low delta T typically indicates poor heat exchange efficiency or system malfunction, which usually requires higher input power to maintain load, not lower. The logical direction is incorrect for identifying an anomaly.",False,1.71,322,False


In [3]:
paper_cases.groupby("case")["matches paper"].agg(["sum", "count"]).rename(columns={"sum": "matches", "count": "runs"})

,matches,runs
case,,
1,3,3
2,1,3


## (b) Made-up rules with our column names: full pipeline

These rules are invented for testing (not mined); "our expectation" is our own judgement from the
control sequence, not ground truth.

| # | Rule | Our expectation |
|---|---|---|
| 1 | high outdoor dry-bulb -> low chilled-water set-point | 0: normal set-point reset |
| 2 | tower fan high + low wet-bulb -> tower leaving water high | 3: high approach, degradation |
| 3 | low load + secondary pump speed high -> low differential pressure | 2: operational (pump/DP control) |

In [4]:
test_rules = [
    {"antecedent": ("OA_TEMP=high",), "consequent": ("CWL_PRI_SW_TEMPSPT=low",),
     "support": 0.21, "confidence": 0.93, "lift": 3.1},
    {"antecedent": ("CT_FAN_SPD_1=high", "OA_TEMP_WB=low"), "consequent": ("CT_SW_TEMP_1=high",),
     "support": 0.04, "confidence": 0.71, "lift": 2.4},
    {"antecedent": ("CWL_SEC_LOAD=low", "CWL_SEC_PM_SPD_1=high"), "consequent": ("CWL_SEC_DP=low",),
     "support": 0.03, "confidence": 0.65, "lift": 1.9},
]
expected_label = {0: 0, 1: 3, 2: 2}

for i, rule in enumerate(test_rules):
    text = ld.rule_to_text(rule)
    print(f"Rule {i}: {text}")
    for line in rag.describe_variables(text):
        print(f"   def: {line}")
    for c in rag.retrieve(text):
        print(f"   {c['score']:.3f} [{c['source']}#{c['id']}] {c['text']}")
    print()

Rule 0: Antecedent: OA_TEMP=high; Consequent: CWL_PRI_SW_TEMPSPT=low; Support: 0.2100; Confidence: 0.9300; Lift: 3.10
   def: OA_TEMP = outdoor air dry-bulb temperature (degF)
   def: CWL_PRI_SW_TEMPSPT = chilled water supply temperature set-point (degF)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   0.588 [02_control_sequence.md#11] Chilled-water supply set-point reset: the chiller leaving-water set-point (CWL_PRI_SW_TEMPSPT) is reset linearly from outdoor dry-bulb temperature, 52 degF when outdoor air is 60 degF or colder down to 42 degF when it is 80 degF or warmer, updated hourly. A low chilled-water set-point in hot weather is therefore expected.
   0.563 [02_control_sequence.md#13] Condenser water set-point: the tower leaving-water set-point (CT_SW_TEMPSPT) equals outdoor wet-bulb temperature + 8 degF approach, with a lower limit of 34 degF. All running tower fans run at the same speed, controlled by a feedback loop to track this set-point.
   0.554 [01_plant_design.md#5] Units in the data: temperatures in degF, water flows in GPM, power in kW, the secondary-loop cooling load (CWL_SEC_LOAD) in W, differential pressure in inH2O; control signals, speeds and valve position are fractions 0-1; status points are 0 = off, 1 = on.
   0.501 [01_plant_design.md#8] Cooling-tower appr

In [5]:
df_rag = ld.classify_all(test_rules, use_rag=True, out_name="llm_test_rules_rag")
df_norag = ld.classify_all(test_rules, use_rag=False, out_name="llm_test_rules_norag")

both = pd.concat([df_rag, df_norag])
both["rule"] = both["antecedent"] + " -> " + both["consequent"]
both["our expectation"] = both["rule_id"].map(expected_label)
both["matches"] = both["label"] == both["our expectation"]
both["latency_s"] = both["latency_s"].round(2)
both[["rule_id", "use_rag", "run", "validity", "label", "our expectation", "matches", "reason",
      "latency_s", "total_tokens", "failed"]].sort_values(["rule_id", "use_rag", "run"])

rules:   0%|          | 0/3 [00:00<?, ?it/s]

saved 9 rows to results\tables\llm_test_rules_rag.csv  (failed: 0)


rules:   0%|          | 0/3 [00:00<?, ?it/s]

saved 9 rows to results\tables\llm_test_rules_norag.csv  (failed: 0)


,rule_id,use_rag,run,validity,label,our expectation,matches,reason,latency_s,total_tokens,failed
0,0,False,1,False,0,0,True,"This rule describes a normal HVAC control logic: when outdoor air temperature is high, the cooling water loop setpoint reduces to prevent overcooling or energy waste. This is standard operational behavior, not an anomaly.",9.92,262,False
1,0,False,2,False,0,0,True,"This rule lacks context and represents a plausible operational state rather than an anomaly. High outside temperature often requires lower cooling setpoints to maintain comfort, so this association does not inherently indicate a design or performance issue.",9.32,261,False
2,0,False,3,False,0,0,True,"The rule describes normal HVAC operation: when outdoor air temperature is high, the cooling water loop setpoint lowers to maximize efficiency and prevent overheating. This is a standard control strategy, not an anomaly.",9.17,260,False
0,0,True,1,True,2,0,False,The high outdoor temperature combined with a low chilled water setpoint indicates the cooling system is operating at full capacity despite extreme heat. This operational pattern suggests potential issues such as excessive fan speeds or valve malfunctions rather than normal design behavior.,13.63,637,False
1,0,True,2,True,2,0,False,"The rule shows high outdoor temp paired with low chilled water setpoint, which matches the described reset logic (expected behavior). However, the extremely high confidence (0.93) and lift (3.1) suggest this pattern is unusually dominant or potentially indicates a control system stuck in an aggressive mode rather than normal operation.",12.59,656,False
2,0,True,3,False,0,0,True,"The rule describes normal operational behavior where chilled water set-points reset lower in high outdoor temperatures to maximize cooling efficiency. This is a standard control strategy, not an anomaly or degradation.",8.57,626,False
3,1,False,1,True,2,3,False,"High fan speed combined with low outdoor wet-bulb temperature typically cools the coil efficiently. A high supply air temperature indicates insufficient cooling performance under these favorable conditions, suggesting an operational issue like dirty coils or flow restriction.",10.18,274,False
4,1,False,2,True,2,3,False,High fan speed combined with low outdoor wet-bulb temperature should not cause high supply air temperature. This indicates a control failure or mechanical fault in the cooling system.,8.27,263,False
5,1,False,3,True,2,3,False,High fan speed combined with low outdoor wet-bulb temperature typically indicates improper control logic or a malfunctioning sensor causing unnecessary energy consumption and potential system inefficiency.,8.13,262,False
3,1,True,1,True,2,3,False,"High fan speed with low approach indicates high energy consumption for poor heat transfer, not necessarily a design flaw or performance degradation yet.",8.26,594,False


In [6]:
summary = both.groupby(["use_rag", "rule_id"]).agg(
    labels=("label", lambda s: list(s)), matches=("matches", "sum"), mean_latency_s=("latency_s", "mean"),
    mean_tokens=("total_tokens", "mean")).round(2)
summary

labels  matches  mean_latency_s  mean_tokens
use_rag rule_id                                                 
False   0        [0, 0, 0]        3            9.47       261.00
        1        [2, 2, 2]        0            8.86       266.33
        2        [2, 2, 2]        3            8.14       261.00
True    0        [2, 2, 0]        1           11.60       639.67
        1        [2, 2, 2]        0            8.68       602.33
        2        [0, 0, 0]        0            8.55       615.33

## (c) Time estimate: 300 rules x 3 runs x 2 settings

From the latencies measured above (RAG prompts are longer, so the two settings are timed separately).
Assumes calls run one after another on this machine.

In [7]:
lat_rag = df_rag["latency_s"].mean()
lat_norag = df_norag["latency_s"].mean()
n_rules, n_runs = 300, ld.SETTINGS["runs"]
calls = n_rules * n_runs
hours = calls * (lat_rag + lat_norag) / 3600
estimate = pd.DataFrame([
    {"setting": "with RAG", "calls": calls, "mean latency (s)": round(lat_rag, 2), "hours": round(calls * lat_rag / 3600, 2)},
    {"setting": "without RAG", "calls": calls, "mean latency (s)": round(lat_norag, 2), "hours": round(calls * lat_norag / 3600, 2)},
    {"setting": "total", "calls": 2 * calls, "mean latency (s)": None, "hours": round(hours, 2)},
])
estimate.to_csv(ROOT / "results/tables/llm_time_estimate.csv", index=False)
estimate

,setting,calls,mean latency (s),hours
0,with RAG,900,9.61,2.40
1,without RAG,900,8.82,2.21
2,total,1800,NaN,4.61


## Notes

**Run 2 (22 Nov 2026), after these changes:** sensor rules in `03_expert_rules.md` made generic, 9 rules on faults
not in our dataset added, `rag.top_k` 3 -> 5, validity sentence added to the prompt. Section 1 was not rerun.

- **Paper cases:** Case 1 now matches 3/3 (was 0/3): the validity sentence fixed the "validity = rule is true" reading.
  Case 2 dropped to 1/3 (was 3/3): in 2 runs the model argued that low delta T with *low* input power is not an
  anomaly and chose label 0.
- **Test rules:** with RAG 1/9 expected labels (run 1: 5/9), without RAG 6/9 (run 1: 3/9).
  - Rule 2 (pump speed high at low load, low DP): with RAG it fell from 3/3 to 0/3. With the generic sensor rule,
    no sensor/pump chunk is in the top 5 any more; the model saw only design and staging text and called it normal.
  - None of the 9 new fault rules were retrieved for these 3 rules (they add realistic distractors without leaking).
  - With only 3 invented rules x 3 runs the differences are mostly noise from a 4B model at temperature 0.7;
    they show variance, not which setting is better.
- **Reasoning off** (section 1, run 1): only `reasoning_effort="none"` works (1.8 s vs 75-89 s);
  `extra_body={"think": False}` and `/no_think` were ignored.
- **Latency varies with machine load** (1.1-14 s per call in run 2). Time estimate for 1,800 calls: about 0.75 h at
  1.5 s/call up to about 4.6 h at the latency measured above. Run the full experiment when nothing else uses the GPU.
- The `ImportError ... NumPy 1.x` messages in the setup cell come from optional pandas add-ons in the Anaconda
  environment used to run this notebook; they are harmless.
- The knowledge base contains general HVAC knowledge and the plant description only, no dataset-specific answers.